# Report Dataset Statistics

> Purpose: generate the dataset statistics used in the report and validate the Dataset class.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

In [2]:
project_src = Path("../src").resolve()
if str(project_src) not in sys.path:
    sys.path.insert(0, str(project_src))

from speech_recognition.dataset.manager import SpeechCommandsDataset  # noqa: E402

In [3]:
root = Path("../data/kaggle_speech_commands/tensorflow-speech-recognition-challenge")
if not (root / "train" / "audio").exists():
    candidates = list(Path("../data/kaggle_speech_commands").rglob("train/audio"))
    if not candidates:
        raise FileNotFoundError("Dataset root not found under ../data/kaggle_speech_commands")
    root = candidates[0].parent.parent

split_dir = root / "train" / "split_lists"
target_commands = {"yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"}


def norm_label(raw_label: str) -> str:
    if raw_label == "_background_noise_":
        return "__silence__"
    if raw_label in target_commands:
        return raw_label
    return "__unknown__"


split_files = [
    ("train_small", "small_training_list.txt", "Phases 1-3 train (10-class)"),
    ("valid_small", "small_validation_list.txt", "Phases 1-3 validation (10-class)"),
    ("test_small", "small_testing_list.txt", "Phases 1-3 diagnostics (10-class)"),
    ("train_extended", "extended_training_list.txt", "Phase 4 train (12-class)"),
    ("valid_extended", "extended_validation_list.txt", "Phase 4 validation (12-class)"),
    ("test_extended", "extended_testing_list.txt", "Phase 4 test (12-class)"),
]

rows = []
for split_name, file_name, used_for in split_files:
    path = split_dir / file_name
    if not path.exists():
        continue

    lines = [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    counts = Counter(norm_label(ln.split("/", 1)[0]) for ln in lines)

    rows.append(
        {
            "split": split_name,
            "total": int(sum(counts.values())),
            "classes_12": int(sum(1 for v in counts.values() if v > 0)),
            "command_total": int(sum(counts[label] for label in target_commands)),
            "unknown": int(counts["__unknown__"]),
            "silence": int(counts["__silence__"]),
            "used_for": used_for,
        }
    )

df_split_summary = pd.DataFrame(rows)
split_order = [name for name, _, _ in split_files]
df_split_summary["split"] = pd.Categorical(
    df_split_summary["split"], categories=split_order, ordered=True
)
df_split_summary = df_split_summary.sort_values("split").reset_index(drop=True)

print("Split summary (report table source):")
display(df_split_summary)

small = df_split_summary[df_split_summary["split"].str.contains("_small")]
extended = df_split_summary[df_split_summary["split"].str.contains("_extended")]

print("\nPolicy checks:")
print(
    "- Small splits are command-only:",
    bool((small[["unknown", "silence"]] == 0).all().all()),
)
print(
    "- Small split totals match 10x quotas (10000/2500/2500):",
    bool(
        small.set_index("split")["total"].to_dict()
        == {"train_small": 10000, "valid_small": 2500, "test_small": 2500}
    ),
)
print(
    "- Extended command and unknown counts are balanced per split:",
    bool((extended["command_total"] == extended["unknown"]).all()),
)

latex_table = df_split_summary[
    [
        "split",
        "total",
        "classes_12",
        "command_total",
        "unknown",
        "silence",
        "used_for",
    ]
].to_latex(index=False, escape=False)
print("\nLaTeX table snippet:")
print(latex_table)

Split summary (report table source):


,split,total,classes_12,command_total,unknown,silence,used_for
0,train_small,10000,10,10000,0,0,Phases 1-3 train (10-class)
1,valid_small,2500,10,2500,0,0,Phases 1-3 validation (10-class)
2,test_small,2500,10,2500,0,0,Phases 1-3 diagnostics (10-class)
3,train_extended,38210,12,18946,18946,318,Phase 4 train (12-class)
4,valid_extended,4776,12,2368,2368,40,Phase 4 validation (12-class)
5,test_extended,4776,12,2368,2368,40,Phase 4 test (12-class)



Policy checks:
- Small splits are command-only: True
- Small split totals match 10x quotas (10000/2500/2500): True
- Extended command and unknown counts are balanced per split: True

LaTeX table snippet:
\begin{tabular}{lrrrrrl}
\toprule
split & total & classes_12 & command_total & unknown & silence & used_for \\
\midrule
train_small & 10000 & 10 & 10000 & 0 & 0 & Phases 1-3 train (10-class) \\
valid_small & 2500 & 10 & 2500 & 0 & 0 & Phases 1-3 validation (10-class) \\
test_small & 2500 & 10 & 2500 & 0 & 0 & Phases 1-3 diagnostics (10-class) \\
train_extended & 38210 & 12 & 18946 & 18946 & 318 & Phase 4 train (12-class) \\
valid_extended & 4776 & 12 & 2368 & 2368 & 40 & Phase 4 validation (12-class) \\
test_extended & 4776 & 12 & 2368 & 2368 & 40 & Phase 4 test (12-class) \\
\bottomrule
\end{tabular}



In [ ]:
TARGET_COMMANDS = {"yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"}
ALLOWED_EXTENDED = TARGET_COMMANDS | {"__unknown__", "__silence__"}


def validate_mode(dataset: SpeechCommandsDataset, mode_name: str, command_only: bool) -> dict:
    train, val, test = dataset.train, dataset.val, dataset.test
    all_samples = train + val + test
    stats = dataset.stats()

    assert len(all_samples) > 0, f"{mode_name}: dataset is empty"
    assert all(sample.path.exists() for sample in all_samples), f"{mode_name}: missing file path(s)"
    assert all(dataset._is_1sec_sample(sample) for sample in all_samples), (
        f"{mode_name}: found non-1-second sample despite only_1sec_samples=True"
    )

    label_counts = Counter(sample.label for sample in all_samples)
    label_set = set(label_counts)

    if command_only:
        unexpected_labels = sorted(label_set - TARGET_COMMANDS)
        assert label_set.issubset(TARGET_COMMANDS), (
            f"{mode_name}: unexpected labels in command-only mode: {unexpected_labels}"
        )
    else:
        unexpected_labels = sorted(label_set - ALLOWED_EXTENDED)
        assert label_set.issubset(ALLOWED_EXTENDED), (
            f"{mode_name}: unexpected labels in extended mode: {unexpected_labels}"
        )
        assert label_counts["__unknown__"] > 0, f"{mode_name}: missing __unknown__ samples"
        assert label_counts["__silence__"] > 0, f"{mode_name}: missing __silence__ samples"

    return {
        "mode": mode_name,
        "stats": stats,
        "labels": dict(sorted(label_counts.items())),
    }


small_ds = SpeechCommandsDataset(
    data_dir_name="../data/kaggle_speech_commands",
    auto_download=False,
    use_smaller_dataset=True,
    use_extended_dataset=False,
    only_1sec_samples=True,
    seed=42,
    unknown_label_samples_size=10000,
)

extended_ds = SpeechCommandsDataset(
    data_dir_name="../data/kaggle_speech_commands",
    auto_download=False,
    use_smaller_dataset=False,
    use_extended_dataset=True,
    only_1sec_samples=True,
    seed=42,
    unknown_label_samples_size=10000,
)

results = [
    validate_mode(small_ds, "small", command_only=True),
    validate_mode(extended_ds, "extended", command_only=False),
]

for result in results:
    print(f"\n[{result['mode']}] split sizes: {result['stats']}")
    print(f"[{result['mode']}] label counts: {result['labels']}")

print("\nDataset class validation checks passed.")

2026-04-14 12:36:15,036 - speech_recognition.dataset.manager - INFO - Saved unknown origin mapping CSV with 78195 rows: /Users/pawelp/Desktop/education/pw/deepl/speech-recognition/src/../data/kaggle_speech_commands/tensorflow-speech-recognition-challenge/train/split_lists/unknown_origin_labels.csv
2026-04-14 12:36:16,872 - speech_recognition.dataset.manager - INFO - Command-only lists saved: train=18946, val=2368, test=2368
2026-04-14 12:36:16,948 - speech_recognition.dataset.manager - INFO - Small command-only lists saved with fixed per-class sizes: train=10000, val=2500, test=2500
2026-04-14 12:36:20,242 - speech_recognition.dataset.manager - INFO - Extended balanced lists saved: train=38210, val=4776, test=4776
2026-04-14 12:36:20,750 - speech_recognition.dataset.manager - INFO - Building splits using official lists (training=10000, validation=2500, testing=2500)
2026-04-14 12:36:21,197 - speech_recognition.dataset.manager - INFO - Normalizing split durations for 1-second mode: shor


[small] split sizes: {'train': 10000, 'val': 2500, 'test': 2500}
[small] label counts: {'down': 1500, 'go': 1500, 'left': 1500, 'no': 1500, 'off': 1500, 'on': 1500, 'right': 1500, 'stop': 1500, 'up': 1500, 'yes': 1500}

[extended] split sizes: {'train': 38210, 'val': 4776, 'test': 4776}
[extended] label counts: {'__silence__': 398, '__unknown__': 23682, 'down': 2359, 'go': 2372, 'left': 2353, 'no': 2375, 'off': 2357, 'on': 2367, 'right': 2367, 'stop': 2380, 'up': 2375, 'yes': 2377}

Dataset class validation checks passed.
